# Day 2: State and Reducers

Yesterday every key in our state was written by exactly one node. Life was simple.

Today we ask a slightly uncomfortable question: **what happens when two nodes write to the same key?**

Picture a shared WhatsApp note that your whole family uses as a shopping list. Your mother writes "milk". Then your brother opens it and, instead of adding to the list, he replaces the whole thing with "chips". Milk is gone. Nobody did anything wrong on purpose; the note just didn't have a rule for *combining* updates.

A **reducer** is that rule. It tells LangGraph how to combine the old value with a new one: replace it, add to it, keep the bigger one, whatever you need.

By the end of today you'll be able to:

- explain why the default behaviour is "overwrite"
- use `operator.add` to make a list grow instead of being replaced
- write your own reducer
- run two nodes at the same time (in parallel) without them fighting
- hide internal "scratch" keys from the final output

No API key needed today either.

In [ ]:
%pip install -q -U langgraph

In [ ]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

In [ ]:
def show_graph(app):
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

## 1. The default rule is "overwrite"

Here's a state with a `log` list. Two nodes run one after another, and both write to `log`.

In [ ]:
class LogState(TypedDict):
    log: list[str]

In [ ]:
def step_a(state):
    return {"log": ["A ran"]}

In [ ]:
def step_b(state):
    return {"log": ["B ran"]}

In [ ]:
def build(state_class):
    # Small helper so we can build the same graph with different state classes
    builder = StateGraph(state_class)
    builder.add_sequence([step_a, step_b])
    builder.add_edge(START, "step_a")
    builder.add_edge("step_b", END)
    return builder.compile()

### Predict before you run

Both nodes write to `log`. After the graph finishes, what's inside `log`?

<details>
<summary>Click to see the answer</summary>

Only `['B ran']`.

By default, a new value **replaces** the old one. So `step_b` overwrote what `step_a` wrote. It's the brother-replacing-the-shopping-list problem.
</details>

In [ ]:
print(build(LogState).invoke({"log": []}))

For a single value like a score or a name, overwriting is exactly what you want. But for a list that should keep growing, like a log or a chat history, it's a problem.

## 2. Reducers: combine instead of overwrite

To attach a reducer to a key, we wrap its type in `Annotated`, like this:

```python
log: Annotated[list[str], operator.add]
```

Read it as: "`log` is a list of strings, and when someone sends a new value, combine it using `operator.add`".

What does `operator.add` do with two lists? The same thing as `+`. And `["A ran"] + ["B ran"]` gives `["A ran", "B ran"]`. That's it, no magic.

In [ ]:
# Just to prove there's nothing special about operator.add:
print(operator.add(["A ran"], ["B ran"]))
print(operator.add(10, 5))

Now the same graph, with one line changed in the state.

In [ ]:
class AppendLogState(TypedDict):
    log: Annotated[list[str], operator.add]     # the only change from before

In [ ]:
result = build(AppendLogState).invoke({"log": []})
print(result)

Both entries survived. That's a reducer.

Since `operator.add` on numbers means plus, the same trick turns a key into a running counter:

In [ ]:
class CounterState(TypedDict):
    total: Annotated[int, operator.add]

In [ ]:
def add_five(state):
    return {"total": 5}

In [ ]:
def add_ten(state):
    return {"total": 10}

In [ ]:
builder = StateGraph(CounterState)

In [ ]:
builder.add_sequence([add_five, add_ten])

In [ ]:
builder.add_edge(START, "add_five")
builder.add_edge("add_ten", END)

### Predict before you run

We start with `total = 100`. What's the final total?

<details>
<summary>Click to see the answer</summary>

`115`. The reducer does 100 + 5 = 105, then 105 + 10 = 115.

Notice the nodes don't need to know the current total at all. They just say "add 5" and the reducer handles it.
</details>

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"total": 100}))

## 3. Writing your own reducer

A reducer is just a function that takes two things, the **old** value and the **new** value, and returns the combined value. You can write any rule you like.

Example: three exams, and we only care about the best score. A lower score should never replace a higher one.

In [ ]:
def keep_highest(old: int, new: int) -> int:
    # Whatever comes in, keep whichever is bigger
    return max(old, new)

In [ ]:
class ExamState(TypedDict):
    best_score: Annotated[int, keep_highest]

In [ ]:
def exam_1(state):
    return {"best_score": 72}

In [ ]:
def exam_2(state):
    return {"best_score": 65}    # a bad day, this should be ignored

In [ ]:
def exam_3(state):
    return {"best_score": 88}

In [ ]:
builder = StateGraph(ExamState)

In [ ]:
builder.add_sequence([exam_1, exam_2, exam_3])

In [ ]:
builder.add_edge(START, "exam_1")
builder.add_edge("exam_3", END)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"best_score": 0}))

### Your turn

Change `keep_highest` so it keeps the **lowest** score instead (careful about the starting value, what happens if you start with 0?). Then run it.

In [ ]:
def keep_lowest(old: int, new: int) -> int:
    return min(old, new)        # this is the only line that changed

In [ ]:
class WorstExamState(TypedDict):
    best_score: Annotated[int, keep_lowest]

In [ ]:
builder = StateGraph(WorstExamState)

In [ ]:
builder.add_sequence([exam_1, exam_2, exam_3])

In [ ]:
builder.add_edge(START, "exam_1")
builder.add_edge("exam_3", END)

In [ ]:
# Try changing 100 to 0 and think about why the answer changes
app = builder.compile()

In [ ]:
print(app.invoke({"best_score": 100}))

## 4. Why reducers really matter: running nodes in parallel

So far one node runs after another. But LangGraph can also run nodes **at the same time**.

If a node has edges going to two different nodes, both of them run together in the same step. Think of it like a manager handing two tasks to two people at once instead of waiting for one to finish. This is called a **fan-out**. When both are done and both point to the same next node, the graph joins back up. That's a **fan-in**.

```
            +--> check_weather --+
  START ----+                    +--> summary --> END
            +--> check_traffic --+
```

Here's the question that makes reducers essential: if `check_weather` and `check_traffic` both write to `notes` *in the same moment*, who wins? Without a rule, LangGraph honestly can't decide, so it refuses.

In [ ]:
from langgraph.errors import InvalidUpdateError

In [ ]:
def make_trip_graph(state_class):
    def check_weather(state):
        return {"notes": ["Weather: sunny"]}

    def check_traffic(state):
        return {"notes": ["Traffic: light"]}

    def summary(state):
        return {"notes": [f"{len(state['notes'])} checks done"]}

    builder = StateGraph(state_class)
    builder.add_node("check_weather", check_weather)
    builder.add_node("check_traffic", check_traffic)
    builder.add_node("summary", summary)

    # Two edges leaving START means both nodes run at the same time
    builder.add_edge(START, "check_weather")
    builder.add_edge(START, "check_traffic")

    # Both feed into summary, which waits for both of them
    builder.add_edge("check_weather", "summary")
    builder.add_edge("check_traffic", "summary")
    builder.add_edge("summary", END)
    return builder.compile()

In [ ]:
class NoReducer(TypedDict):
    notes: list[str]

In [ ]:
try:
    make_trip_graph(NoReducer).invoke({"notes": []})
except InvalidUpdateError as error:
    print("LangGraph refused:", type(error).__name__)

Now give `notes` a reducer and try again.

In [ ]:
class WithReducer(TypedDict):
    notes: Annotated[list[str], operator.add]

In [ ]:
trip = make_trip_graph(WithReducer)

In [ ]:
show_graph(trip)

In [ ]:
print(trip.invoke({"notes": []}))

A simple rule to remember: **if more than one node can write to a key in the same step, that key needs a reducer.**

## 5. When a key might not be there yet

You don't have to pass every key when you call `invoke`. But if a node tries to read a key that was never set, Python raises a `KeyError`, the same as with any normal dictionary.

The fix is the one you already know from Python: use `state.get("key", default)`.

In [ ]:
class Visit(TypedDict):
    name: str
    visits: int

In [ ]:
def count_visit(state: Visit) -> dict:
    # If "visits" was never passed in, treat it as 0
    return {"visits": state.get("visits", 0) + 1}

In [ ]:
builder = StateGraph(Visit)

In [ ]:
builder.add_node("count_visit", count_visit)

In [ ]:
builder.add_edge(START, "count_visit")
builder.add_edge("count_visit", END)

In [ ]:
visit_app = builder.compile()

In [ ]:
print(visit_app.invoke({"name": "Meera"}))               # visits was not given
print(visit_app.invoke({"name": "Meera", "visits": 4}))

You can also use a **Pydantic model** as your state instead of a `TypedDict`. The nice thing is you get default values and type checking for free. The one thing to remember is that you read fields with a dot (`state.qty`), not with square brackets.

In [ ]:
from pydantic import BaseModel

In [ ]:
class Order(BaseModel):
    item: str
    qty: int = 1          # if nobody passes qty, it's 1

In [ ]:
def describe(state: Order) -> dict:
    return {"item": f"{state.qty} x {state.item}"}

In [ ]:
builder = StateGraph(Order)

In [ ]:
builder.add_node("describe", describe)

In [ ]:
builder.add_edge(START, "describe")
builder.add_edge("describe", END)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"item": "masala dosa"}))

## 6. Hiding internal keys from the output

Sometimes nodes need scratch space, like rough notes you write while solving a problem, which the user doesn't need to see. You can give the graph a separate **input schema** (what goes in) and **output schema** (what comes out). Everything else stays internal.

In [ ]:
class InputState(TypedDict):
    question: str

In [ ]:
class OutputState(TypedDict):
    answer: str

In [ ]:
class FullState(InputState, OutputState):
    scratch_notes: str        # used inside, never shown outside

In [ ]:
def think(state: FullState) -> dict:
    return {"scratch_notes": f"The user is asking about: {state['question']}"}

In [ ]:
def respond(state: FullState) -> dict:
    return {"answer": f"Here is my answer to '{state['question']}'"}

In [ ]:
builder = StateGraph(FullState, input_schema=InputState, output_schema=OutputState)

In [ ]:
builder.add_sequence([think, respond])

In [ ]:
builder.add_edge(START, "think")
builder.add_edge("respond", END)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"question": "What is LangGraph?"}))

Only `answer` came back. `scratch_notes` did its job and stayed hidden.

## 7. Common mistakes

**Mistake 1: Returning the whole list when the key has a reducer.**
This one is sneaky. With `operator.add`, whatever you return gets *added* to what's already there. So if you return the old list plus your new item, you double everything.

In [ ]:
class Items(TypedDict):
    items: Annotated[list[str], operator.add]

In [ ]:
def wrong_way(state: Items) -> dict:
    # Looks reasonable, but the reducer will ADD this whole list to the existing one
    return {"items": state["items"] + ["tea"]}

In [ ]:
builder = StateGraph(Items)

In [ ]:
builder.add_node("wrong_way", wrong_way)

In [ ]:
builder.add_edge(START, "wrong_way")
builder.add_edge("wrong_way", END)

In [ ]:
app = builder.compile()

In [ ]:
print(app.invoke({"items": ["milk", "bread"]}))
# milk and bread now appear twice

**Fix:** with a reducer, return only the new part: `return {"items": ["tea"]}`.

**Mistake 2: Forgetting to wrap the value in a list.**
`operator.add` on a list and a string fails, because Python can't do `["milk"] + "tea"`.

In [ ]:
def forgot_the_list(state: Items) -> dict:
    return {"items": "tea"}         # should be ["tea"]

In [ ]:
builder = StateGraph(Items)

In [ ]:
builder.add_node("forgot_the_list", forgot_the_list)

In [ ]:
builder.add_edge(START, "forgot_the_list")
builder.add_edge("forgot_the_list", END)

In [ ]:
try:
    app = builder.compile()
    app.invoke({"items": ["milk"]})
except TypeError as error:
    print("TypeError:", error)

**Fix:** always return a list for a list key, even if it has one item.

**Mistake 3: Parallel nodes writing the same key without a reducer.** You saw this in section 4. If you get `InvalidUpdateError`, look for a key that two nodes write to in the same step, and give it a reducer.

## 8. Mini project: a shopping cart

Let's put it together. Three "shelves" add items to the cart at the same time, and a checkout node adds up the bill.

In [ ]:
class Cart(TypedDict):
    items: Annotated[list[str], operator.add]
    total: Annotated[int, operator.add]
    bill: str

In [ ]:
def fruits(state):
    return {"items": ["apple"], "total": 120}

In [ ]:
def dairy(state):
    return {"items": ["milk", "paneer"], "total": 60 + 90}

In [ ]:
def snacks(state):
    return {"items": ["chips"], "total": 20}

In [ ]:
def checkout(state):
    return {"bill": f"{len(state['items'])} items, Rs. {state['total']}"}

In [ ]:
builder = StateGraph(Cart)

In [ ]:
for fn in (fruits, dairy, snacks, checkout):
    builder.add_node(fn.__name__, fn)

In [ ]:
for shelf in ("fruits", "dairy", "snacks"):
    builder.add_edge(START, shelf)        # all three shelves run in parallel
    builder.add_edge(shelf, "checkout")   # and checkout waits for all of them
builder.add_edge("checkout", END)

In [ ]:
cart_app = builder.compile()
show_graph(cart_app)

### Predict before you run

How many items, and what total?

<details>
<summary>Click to see the answer</summary>

4 items (apple, milk, paneer, chips) and Rs. 290 (120 + 150 + 20).

The order of the items might be different on your machine, because the three shelves run at the same time. The total is always the same.
</details>

In [ ]:
result = cart_app.invoke({"items": [], "total": 0})
print(result)
assert result["bill"] == "4 items, Rs. 290"

## 9. Practice

**Exercise 1.** Write a reducer called `add_unique(old, new)` that adds new items to the list, but skips anything that's already in it. Use it for a `tags` key. Two nodes both add `"python"`, and it should only appear once.

**Exercise 2.** Three judges score a performance at the same time: 7, 9 and 8. Collect all the scores in a list, then an `average` node calculates the mean. You should get 8.0.

In [ ]:
# Solution 1
def add_unique(old: list, new: list) -> list:
    return old + [tag for tag in new if tag not in old]

In [ ]:
class TagState(TypedDict):
    tags: Annotated[list[str], add_unique]

In [ ]:
def tag_a(state):
    return {"tags": ["python", "ai"]}

In [ ]:
def tag_b(state):
    return {"tags": ["python", "langgraph"]}

In [ ]:
builder = StateGraph(TagState)

In [ ]:
builder.add_sequence([tag_a, tag_b])

In [ ]:
builder.add_edge(START, "tag_a")
builder.add_edge("tag_b", END)

In [ ]:
app = builder.compile()

In [ ]:
result = app.invoke({"tags": []})
print(result)
assert result["tags"] == ["python", "ai", "langgraph"]

In [ ]:
# Solution 2
class JudgeState(TypedDict):
    scores: Annotated[list[int], operator.add]
    average: float

In [ ]:
def judge_1(state):
    return {"scores": [7]}

In [ ]:
def judge_2(state):
    return {"scores": [9]}

In [ ]:
def judge_3(state):
    return {"scores": [8]}

In [ ]:
def average(state):
    return {"average": sum(state["scores"]) / len(state["scores"])}

In [ ]:
builder = StateGraph(JudgeState)

In [ ]:
for fn in (judge_1, judge_2, judge_3, average):
    builder.add_node(fn.__name__, fn)

In [ ]:
for judge in ("judge_1", "judge_2", "judge_3"):
    builder.add_edge(START, judge)
    builder.add_edge(judge, "average")
builder.add_edge("average", END)

In [ ]:
app = builder.compile()

In [ ]:
result = app.invoke({"scores": []})
print(result)
assert result["average"] == 8.0

## Wrapping up

The big idea today: by default, a new value **replaces** the old one. A reducer changes that rule. `Annotated[list, operator.add]` makes a list grow, and you can write any rule you want as a plain two-argument function.

Reducers become essential the moment nodes run in parallel, because two updates arriving at the same time need a rule for combining them.

Tomorrow the graph starts making **decisions**: going left or right depending on the data, and looping until a job is done. That's the part that turns a pipeline into something that feels smart.